# Curso APIs y Web Scraping — Laboratorio en vivo
### Capítulo 3: Fundamentos de Web Scraping · Capítulo 4: Técnicas avanzadas

Ejecuta cada celda con **Shift + Enter**.

> **No necesitas instalar nada.** Google Colab ya trae `requests`, `beautifulsoup4`,
> `lxml` y `pandas`. Si lo abres en tu PC y algo falla, corre la celda de preparación.

**Sitio de práctica:** [books.toscrape.com](https://books.toscrape.com) — una librería
falsa creada justamente para aprender scraping. Practicar aquí, y no en una web real,
es la primera aplicación de todo lo que vimos en el bloque de ética.

In [ ]:
# Colab ya trae todo esto. Si estás en local y falla, descomenta:
# !pip install -q requests beautifulsoup4 lxml pandas

import json, re, time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("requests", requests.__version__, "· pandas", pd.__version__)
print("Todo listo ✔")

---
# Capítulo 3 · Bloque 1
## 1. Pedir una página es la MISMA petición del capítulo 1

Cambia una sola cosa: lo que vuelve. Antes era JSON ordenado; ahora es HTML crudo.

In [ ]:
URL = "https://books.toscrape.com/"

# Identificarse es la primera regla de cortesía del scraping.
HEADERS = {"User-Agent": "BSG-Curso-Scraping/1.0 (ejercicio academico; contacto@ejemplo.com)"}

r = requests.get(URL, headers=HEADERS, timeout=10)

print("status_code  :", r.status_code)
print("Content-Type :", r.headers["Content-Type"])   # ← ¡ya no dice application/json!
print("Tamaño       :", f"{len(r.text):,}", "caracteres")

`text/html` en vez de `application/json`: nos mandaron una **página**, no datos.

Por eso aquí **`r.json()` explota**. Lo que hay es texto con etiquetas:

In [ ]:
print(r.text[:400])

### La trampa del encoding

El servidor dice que la página está en un formato y en realidad está en otro.
Resultado: tildes y símbolos rotos (`Â£`, `PerÃº`, `Ã©`).

In [ ]:
precio_roto = r.text.split('price_color">')[1].split("<")[0]
print("requests cree que es :", r.encoding)
print("pero en realidad es  :", r.apparent_encoding)
print("precio mal leído     :", repr(precio_roto))

r.encoding = r.apparent_encoding          # ← la corrección, UNA línea

precio_ok = r.text.split('price_color">')[1].split("<")[0]
print("precio bien leído    :", repr(precio_ok))

### ¿Quién dice el servidor que eres? El User-Agent

`httpbin.org` es un servicio que te devuelve lo que le enviaste. Sirve para ver
tu propia petición desde el otro lado.

In [ ]:
sin_ua = requests.get("https://httpbin.org/user-agent", timeout=10).json()
con_ua = requests.get("https://httpbin.org/user-agent", headers=HEADERS, timeout=10).json()

print("Sin headers, el servidor ve:", sin_ua["user-agent"])
print("Con headers, el servidor ve:", con_ua["user-agent"])

`python-requests/2.x` es una bandera roja para cualquier sitio.
Poner un User-Agent propio **no es disfrazarse: es presentarse**.

### robots.txt — el cartel de la entrada

Ojo al leerlo: un `robots.txt` tiene **bloques por robot**. Un `Disallow: /` puede estar
dirigido a un bot específico, no a ti. Lo que te aplica es el bloque `User-agent: *`.

In [ ]:
def reglas_para_todos(texto):
    """Devuelve las reglas Disallow del bloque 'User-agent: *'."""
    reglas, dentro = [], False
    for linea in texto.splitlines():
        limpia = linea.split("#")[0].strip()
        if limpia.lower().startswith("user-agent:"):
            dentro = limpia.split(":", 1)[1].strip() == "*"
        elif dentro and limpia.lower().startswith("disallow:"):
            reglas.append(limpia)
    return reglas


for sitio in ["https://www.gob.pe", "https://www.python.org"]:
    rb = requests.get(f"{sitio}/robots.txt", headers=HEADERS, timeout=10)
    print(f"\n{sitio}/robots.txt → {rb.status_code}")
    for regla in reglas_para_todos(rb.text)[:4]:
        print("   ", regla)

In [ ]:
# Python trae un lector de robots.txt en la librería estándar:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://www.python.org/robots.txt")
rp.read()

print("¿Puedo entrar a /downloads/ ?", rp.can_fetch("*", "https://www.python.org/downloads/"))
print("¿Puedo entrar a /webstats/ ?", rp.can_fetch("*", "https://www.python.org/webstats/"))

### El 403: cuando te dicen que no

El servidor **entendió** tu petición y se niega. No es un error de sintaxis.

In [ ]:
bloqueado = requests.get("https://httpbin.org/status/403", timeout=10)
print("status_code:", bloqueado.status_code, "· r.ok:", bloqueado.ok)

**Qué hacer ante un 403, en orden:**
1. Revisar `robots.txt` — quizá esa ruta está prohibida.
2. Identificarte con un User-Agent honesto.
3. Bajar la frecuencia: `time.sleep()` entre peticiones.
4. Buscar si hay API oficial o descarga en CSV.
5. Escribir al sitio y pedir permiso.

❌ Lo que **no** hacemos: rotar IPs, resolver CAPTCHAs, falsear identidad.

> **Nota sobre Colab:** este notebook corre en un servidor de Google, no en tu PC.
> Algunos sitios responden 403 a las IPs de centros de datos aunque desde tu laptop
> abran perfectamente. Si te pasa, no es tu código: es exactamente el bloqueo del que
> habla esta diapositiva.

---
# Capítulo 3 · Bloque 3
## 2. Leer el HTML con BeautifulSoup

`requests` es el **cartero**: trae el sobre. `BeautifulSoup` es la **lupa**: lo lee.

Empecemos con el mismo ejemplo de la diapositiva:

In [ ]:
HTML_CLASE = """
<body>
  <div id="saludo" class="contenedor">
    <p class="texto">Buenas noches</p>
    <span class="texto">Hola nuevamente!</span>
    <a href="https://bsg.edu.pe/curso">Ver el curso</a>
  </div>
</body>
"""

sopa = BeautifulSoup(HTML_CLASE, "html.parser")
div = sopa.find("div")

print("El <div> es hijo de     :", f"<{div.parent.name}>")
print("Hijos directos del <div>:", [h.name for h in div.find_all(recursive=False)])
print("Texto del <p>           :", repr(sopa.find("p").text))
print("Enlace del <a>          :", sopa.find("a")["href"])

### `find()` trae UNO · `find_all()` trae TODOS

In [ ]:
print("find('p')               →", sopa.find("p"))
print("find_all(class_='texto') →", len(sopa.find_all(class_="texto")), "elementos")

# ⚠️ Se escribe class_ CON GUION BAJO: 'class' es palabra reservada de Python.
# ⚠️ find() devuelve None si no encuentra nada (no da error):
print("find('table')           →", sopa.find("table"))

Ese `None` es el origen del error más común de todos:

```
AttributeError: 'NoneType' object has no attribute 'text'
```

Siempre valida antes de pedir `.text`.

### Los atributos: `class` (grupo) · `id` (único) · `href` (a dónde lleva)

In [ ]:
print("div['id']             →", repr(div["id"]), "  (único en la página)")
print("div['class']          →", div["class"], "  ← ¡es una LISTA!")
print("div.get('data-precio')→", div.get("data-precio"), "  (get no explota si no existe)")

### Ahora sí, la página real

In [ ]:
soup = BeautifulSoup(r.text, "html.parser")

libros = soup.find_all("article", class_="product_pod")
print("Libros encontrados:", len(libros))

# ¿Por qué funciona? Porque los 20 libros COMPARTEN la misma clase.
# Eso es exactamente para lo que sirve 'class': marcar un grupo.

for libro in libros[:5]:
    titulo = libro.h3.a["title"]                        # el título completo está en el atributo
    precio = libro.select_one("p.price_color").text
    print(f"{titulo[:45]:<45} {precio:>9}")

### La otra forma: `select()` con selectores CSS

| Quiero… | `find` | `select` (CSS) |
|---|---|---|
| Todos los `<p>` | `find_all("p")` | `select("p")` |
| Por clase | `find_all("div", class_="caja")` | `select("div.caja")` |
| Por id | `find(id="total")` | `select_one("#total")` |
| Anidado | — | `select("article.product_pod h3 a")` |

In [ ]:
titulos = soup.select("article.product_pod h3 a")
print(len(titulos), "títulos · el primero:", titulos[0]["title"])

# href: el atributo que permite SEGUIR NAVEGANDO
print("\nEnlaces al detalle:")
for a in titulos[:3]:
    print("   ", a["href"])

siguiente = soup.select_one("li.next a")
print("\nEnlace a la página siguiente:", siguiente["href"])

---
## 3. XPath: la ruta hasta el dato

XPath es la **ruta de carpetas** de una página web:

```
C:/Usuarios/Bryan/Documentos     ← ruta de archivos
/html/body/div/p                 ← ruta de etiquetas
```

| Símbolo | Significa | Ejemplo |
|---|---|---|
| `/` | hijo directo | `/body/div` |
| `//` | en cualquier nivel ← **el más usado** | `//p` |
| `@` | atributo | `//div[@class='titulo']` |
| `[ ]` | condición o posición | `//tr[2]` |
| `text()` | el texto del nodo | `//h1/text()` |
| `contains()` | coincidencia parcial | `//div[contains(@class,'precio')]` |

In [ ]:
from lxml import html as lxml_html

EJEMPLO = """
<body>
  <div id="cabecera"><h1>Curso BSG</h1></div>
  <div class="lista">
    <p class="precio">S/ 120.00</p>
    <p class="precio oferta">S/ 89.90</p>
    <a href="/detalle/1">Ver más</a>
  </div>
</body>
"""

arbol = lxml_html.fromstring(EJEMPLO)

for expr in ["//p", "//div/p", "//p[@class='precio']",
             "//p[contains(@class,'precio')]", "//h1/text()", "//a/@href"]:
    res = arbol.xpath(expr)
    limpio = [x.text_content().strip() if hasattr(x, "text_content") else str(x) for x in res]
    print(f"{expr:<32} → {limpio}")

Fíjate en la diferencia clave:

- `@class='precio'` **no** encuentra al que tiene `class="precio oferta"`
- `contains(@class,'precio')` **sí** lo encuentra

En webs modernas un elemento suele tener 4 o 5 clases juntas. Por eso `contains()` te salva.

### La trampa del "Copy full XPath" de Chrome

In [ ]:
doc = lxml_html.fromstring(r.text)

absoluto = "/html/body/div/div/div/div/section/div[2]/ol/li[1]/article/h3/a/@title"
relativo = "//article[@class='product_pod']//h3/a/@title"

print("Lo que copia Chrome :", doc.xpath(absoluto)[:1])
print("Lo que deberías usar:", doc.xpath(relativo)[:1])
print("\nTotal con el relativo:", len(doc.xpath(relativo)), "títulos")

Los dos devuelven lo mismo **hoy**. Pero si mañana el diseñador agrega un `<div>` arriba,
el absoluto apunta a otra cosa y tu scraper **miente en silencio**. El relativo sigue funcionando.

★ Usa Chrome para **descubrir** el elemento, no para copiar la ruta.

---
# Capítulo 4 · Limpieza y almacenamiento
## 4. El dato scrapeado SIEMPRE llega sucio

Este es el paso que no aparece en los tutoriales y se lleva la mitad del tiempo real.

In [ ]:
crudos = []
for libro in soup.select("article.product_pod"):
    crudos.append({
        "titulo": libro.h3.a["title"],
        "precio": libro.select_one("p.price_color").text,
        "rating": libro.select_one("p.star-rating")["class"],
        "stock":  libro.select_one("p.instock").text,
        "enlace": libro.h3.a["href"],
    })

print("Así llega el primer libro:\n")
for k, v in crudos[0].items():
    print(f"   {k:<8} = {v!r}")

Mira los tipos: **todo es texto**. El precio no se puede sumar, el rating es una lista
de clases CSS y el stock trae saltos de línea.

### Una función de limpieza por campo

In [ ]:
PALABRAS_RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def precio_a_float(texto):
    """'£51.77' → 51.77"""
    numero = re.sub(r"[^\d.]", "", texto)      # deja solo dígitos y el punto
    return float(numero) if numero else None


def rating_a_numero(clases):
    """['star-rating', 'Three'] → 3"""
    for c in clases:
        if c in PALABRAS_RATING:
            return PALABRAS_RATING[c]
    return None


def hay_stock(texto):
    """'\\n In stock \\n' → True"""
    return "in stock" in texto.strip().lower()


e = crudos[0]
print("precio_a_float:", precio_a_float(e["precio"]))
print("rating_a_numero:", rating_a_numero(e["rating"]))
print("hay_stock:", hay_stock(e["stock"]))

Funciones cortas y con nombre claro: cuando el sitio cambie, arreglas **una función**
y no 300 líneas.

### Aplicar la limpieza y armar la tabla

In [ ]:
limpios = [{
    "titulo": c["titulo"].strip(),
    "precio_gbp": precio_a_float(c["precio"]),
    "rating": rating_a_numero(c["rating"]),
    "en_stock": hay_stock(c["stock"]),
    "url": urljoin(URL, c["enlace"]),
} for c in crudos]

df = pd.DataFrame(limpios)
print(df.dtypes, "\n")          # ← esto es lo que ganamos al limpiar
df.head()

In [ ]:
# Y ahora YA se puede analizar:
print("Precio promedio :", round(df["precio_gbp"].mean(), 2))
print("Rating promedio :", round(df["rating"].mean(), 2), "estrellas")
print("Con stock       :", int(df["en_stock"].sum()), "de", len(df))

df.nlargest(3, "precio_gbp")[["titulo", "precio_gbp", "rating"]]

### Puente con el capítulo 2: scraping + API juntos

Los precios están en libras. Para pasarlos a soles **no se scrapea: se usa una API**.
Cada herramienta para lo suyo.

In [ ]:
tc = requests.get("https://open.er-api.com/v6/latest/GBP", timeout=10)
gbp_pen = tc.json()["rates"]["PEN"]

df["precio_pen"] = (df["precio_gbp"] * gbp_pen).round(2)
print(f"1 GBP = {gbp_pen} PEN")
df[["titulo", "precio_gbp", "precio_pen"]].head()

Este es el patrón profesional: **el HTML da lo que no tiene API, la API da lo que no
debería scrapearse**. Se combinan.

### Guardar y descargar el archivo

En Colab los archivos se guardan en un servidor temporal. Para bajarlos a tu PC:

In [ ]:
df.to_csv("libros.csv", index=False, encoding="utf-8-sig")
# ⚠️ utf-8-sig, NO utf-8: es lo que hace que Excel muestre bien las tildes.

try:
    from google.colab import files
    files.download("libros.csv")          # ← descarga a tu PC
except ImportError:
    print("Estás en local: el archivo quedó junto al notebook.")

Otras salidas, igual de cortas:

```python
df.to_json("libros.json", orient="records", indent=2, force_ascii=False)
df.to_excel("libros.xlsx", index=False)
df.to_sql("libros", con=engine, if_exists="append", index=False)   # base de datos
```

---
## 5. Paginación: el paso 5 del ciclo

Un scraper de una sola página es un ejercicio. **Seguir enlaces** lo convierte en un
proyecto... y también es lo que puede tumbar un servidor. Por eso los tres frenos:

In [ ]:
MAX_PAGINAS = 3        # nunca dejes un while sin tope
PAUSA = 1.0            # cortesía mínima entre peticiones

sesion = requests.Session()               # reutiliza la conexión: más rápido y menos carga
sesion.headers.update(HEADERS)

url, pagina, todos = URL, 1, []

while url and pagina <= MAX_PAGINAS:
    resp = sesion.get(url, timeout=10)
    resp.encoding = resp.apparent_encoding
    s = BeautifulSoup(resp.text, "html.parser")

    tarjetas = s.select("article.product_pod")
    for t in tarjetas:
        todos.append({
            "titulo": t.h3.a["title"],
            "precio_gbp": precio_a_float(t.select_one("p.price_color").text),
            "url": urljoin(url, t.h3.a["href"]),      # ← href relativo → URL completa
        })
    print(f"Página {pagina}: {len(tarjetas)} libros (acumulado {len(todos)})")

    sig = s.select_one("li.next a")
    url = urljoin(url, sig["href"]) if sig else None
    pagina += 1
    if url:
        time.sleep(PAUSA)                             # ← la pausa NUNCA se negocia

print("\nTotal:", len(todos), "libros")

`urljoin()` es la pieza que casi todos olvidan: el `href` del HTML suele ser relativo
(`catalogue/page-2.html`).

**El costo:** 3 páginas = 3 peticiones. Las 50 páginas del sitio serían 50 peticiones y
casi un minuto con pausa de 1 segundo. Para las 1000 fichas de detalle serían ~18 minutos.
**Ahí** es donde Scrapy (concurrencia) se justifica.

---
## 6. ¿Y Scrapy? Aquí no corre — y eso es justo lo que vimos en la diapositiva

Scrapy levanta su propio motor de eventos y Colab/Jupyter ya tienen el suyo. Dos motores
en el mismo proceso dan el error clásico `ReactorNotRestartable`.

Scrapy se ejecuta **desde la terminal**:

```bash
pip install scrapy
scrapy runspider libros_spider.py -o libros.csv

# Para probar selectores antes de escribir el spider:
scrapy shell "https://books.toscrape.com/"
>>> response.css("article.product_pod h3 a::attr(title)").getall()
```

Y el spider completo — **lo mismo que hicimos arriba, sin bucle, sin `sleep`, sin `urljoin`**:

```python
import scrapy

class LibrosSpider(scrapy.Spider):
    name = "libros"
    start_urls = ["https://books.toscrape.com/"]
    custom_settings = {"ROBOTSTXT_OBEY": True, "DOWNLOAD_DELAY": 1.0,
                       "CLOSESPIDER_PAGECOUNT": 3}

    def parse(self, response):
        for libro in response.css("article.product_pod"):
            yield {
                "titulo": libro.css("h3 a::attr(title)").get(),
                "precio_gbp": libro.css("p.price_color::text").re_first(r"[\d.]+"),
            }
        siguiente = response.css("li.next a::attr(href)").get()
        if siguiente:
            yield response.follow(siguiente, callback=self.parse)
```

Está completo en `scrapy_demo/libros_spider.py` del repositorio.

---
## Checklist de un scraper que no te va a dar vergüenza

- [ ] ¿Revisé si hay **API** antes de escribir esto?
- [ ] ¿Leí el **robots.txt**?
- [ ] ¿Puse **User-Agent** identificable?
- [ ] ¿Puse **timeout** en cada petición?
- [ ] ¿Puse **time.sleep()** entre peticiones?
- [ ] ¿Mi bucle tiene un **tope** o puede correr para siempre?
- [ ] ¿Manejo el **403**, el **429** y la caída de red sin que el script explote?
- [ ] ¿**Documenté** qué scrapeé, de dónde y cuándo?

---

## Tarea

Abre el notebook **`reto_scraping_colab.ipynb`**: sacar frases, autores y etiquetas de
[quotes.toscrape.com](https://quotes.toscrape.com), recorrer 3 páginas y guardar en CSV.